# 12 · Unsupervised learning: group similar entities
No labels here. **Clustering** groups entities with similar behavior so you can manage each group differently (a segmentation, a classic data-science deliverable).

In [ ]:
import sys
sys.path.insert(0, "..")          # lets the notebook import common.py from python_practice/
from common import *              # load_operations, check, OUT, ...
import numpy as np
import pandas as pd

df = load_operations()            # reads the dashboard API if it is running, else data_samples/operations_clean.csv
df["not_completed"] = (df["status"] != "Completed").astype(int)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

## 1. Build one row per entity (a feature table)
Choosing features is the real work. Ratios use sums (non-additive measures again).

In [ ]:
g = df.groupby("entity_name")
prof = pd.DataFrame({
    "avg_revenue": g["revenue"].mean(),
    "avg_cost": g["operational_cost"].mean(),
    "margin": g["profit"].sum() / g["revenue"].sum(),
    "avg_duration": g["duration_minutes"].mean(),
    "problem_rate": g["not_completed"].mean(),
    "weekend_share": df[df["is_weekend"]].groupby("entity_name")["revenue"].sum() / g["revenue"].sum(),
})
prof.round(3)

## 2. Scale, then cluster
K-means uses distances, so a column measured in hundreds (revenue) would drown one measured in fractions (margin). **Always scale first.**

In [ ]:
X = StandardScaler().fit_transform(prof)
sil = {}
for k in range(2, 6):
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    sil[k] = silhouette_score(X, labels)
pd.Series(sil, name="silhouette").round(3)

**Silhouette** runs from −1 to 1: how well each entity fits its own cluster vs the nearest other one. Higher is better. It is a guide, not a verdict. A segmentation must also make business sense.

### Your turn 1
Which `k` has the highest silhouette? Store it in `best_k`.

In [ ]:
best_k = None

In [ ]:
check("best_k", best_k == max(sil, key=sil.get), "max(sil, key=sil.get)")

### Your turn 2: describe the clusters
Fit KMeans with your `best_k` (`n_init=10, random_state=0`), store the labels as a pandas Series indexed by entity name in `segments`, then print `prof.groupby(segments).mean()` and write down, in plain words, what makes each group different.

In [ ]:
segments = None

In [ ]:
if segments is None:
    print("(fit KMeans above first)")
else:
    ok = len(segments) == len(prof) and segments.nunique() == best_k
    check("one segment per entity, best_k segments", ok, "pd.Series(labels, index=prof.index)")
    if ok:
        print(prof.groupby(segments).mean().round(2))

In [ ]:
import matplotlib.pyplot as plt
if segments is not None:
    xy = PCA(n_components=2, random_state=0).fit_transform(X)
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.scatter(xy[:, 0], xy[:, 1], c=segments.to_numpy(), cmap="tab10", s=80)
    for (x, y), name in zip(xy, prof.index): ax.annotate(name, (x, y), fontsize=7, xytext=(4, 4), textcoords="offset points")
    ax.set_title("Entities in 2D (PCA), coloured by segment"); ax.set_xlabel("component 1"); ax.set_ylabel("component 2"); plt.show()

**Caveats to state in a real project:** only 10 entities, so treat this as exploration. The segments depend on which features you chose and how you scaled them. Name each segment for what the business should *do* with it.